## Package imports

In [ ]:
import numpy as np

from sensors.simple_sensors import base_IMU, simple_GPS

from robots.robot_state import INS_robot_state
from robots.simple_quad import simple_quad

from filters.filter_state import INS_filter_state
from filters.filter_covariance import INS_filter_covariance
from filters.GPS_only_INS_EKF import GPS_only_INS_EKF
from filters.GPS_only_INS_EqF import GPS_only_INS_EqF

from simulations.GPS_only_INS_sim import GPS_only_INS_sim

from util import pose_utils, quat_utils

## Environment definition

In [ ]:
# parameters for operational area
g = -9.81*np.array([[0.],[0.],[1.]])            # gravity vector

## Robot configuration

In [ ]:
# configure simple_quad robot quad_1

# wayposes for standard inspection task
quad_1_wayposes: list[pose_utils.Pose] = []
num_wayp = 8

# initial pose
waypos = np.array([[0.],[0.],[0.]])
wayquat = np.array([[1.],[0.],[0.],[0.]])
waypose = pose_utils.Pose(wayquat, waypos)
quad_1_wayposes.append(waypose)

# vertical takeoff to altitude 10
waypos = np.array([[0.],[0.],[10.]])
wayquat = quat_utils.ang_vel2quat(np.pi/3 * np.array([[0.],[0.],[1.]]), 1.)
waypose = pose_utils.Pose(wayquat, waypos)
quad_1_wayposes.append(waypose)

# sideways move
waypos = np.array([[40.],[20.],[10.]])
wayquat = quat_utils.ang_vel2quat(np.pi/9 * np.array([[0.],[0.],[1.]]), 1.)
waypose = pose_utils.Pose(wayquat, waypos)
quad_1_wayposes.append(waypose)

for i in range(num_wayp-2):
    # wayposes designed to do a size 30 square at altitude 20
    waypos = np.array([[80. - 30.*np.cos(i*np.pi/2)],[80. + 30.*np.sin(i*np.pi/2)],[20.]])
    wayquat = quat_utils.ang_vel2quat(np.arctan2(85.-waypos[1,0],75.-waypos[0,0]) * np.array([[0.],[0.],[1.]]), 1.)
    waypose = pose_utils.Pose(wayquat, waypos)
    quad_1_wayposes.append(waypose)

# parameters for simple_quad quad_1
quad_1_params = simple_quad.parameters()
quad_1_params.wayposes = quad_1_wayposes
quad_1_params.prox_lim = 1.0               # how close to a waypose is close enough (in m)

quad_1_params.g = g                        # gravity vector

quad_1_params.d_ab = 0.001                 # accelerometer bias drift in each axis (in m/s^3)
quad_1_params.d_wb = np.deg2rad(.01)       # gyro bias drift in each axis (in rad/s^2)

# configuration flags for simple_quad quad_1
quad_1_config = simple_quad.configuration()

## Sensor configuration

In [ ]:
# configure sensors

# IMU (3-axis accelerometer, 3-axis gyroscope, magnetometer)
IMU_params = base_IMU.parameters()
IMU_params.freq = 100.0                 # measurement frequency of accelerometer and gyroscope in Hz (MUST be an integer multiple of freq_m)
IMU_params.s_a = 0.1                    # uncertainty of accelerometer measurement a_m in each axis (std.dev. in m/s^2)
IMU_params.s_w = np.deg2rad(0.5)        # uncertainty of gyro measurement w_m in each axis (std.dev. in rad/s)

# GPS position
GPS_params = simple_GPS.parameters()
GPS_params.freq = 1.0                   # measurement frequency of GPS in Hz
GPS_params.s_gps = 0.3                  # uncertainty of gps position measurement y_gps in each component (std.dev. in m)

## Filter configuration

In [ ]:
# common filter parameters (use true values -> perfect tuning)

s_a = IMU_params.s_a            # estimate of uncertainty of accelerometer measurement a_m in each axis (std.dev. in m/s^2)
s_w = IMU_params.s_w            # estimate of uncertainty of gyro measurement w_m in each axis (std.dev in rad/s)

s_ab = quad_1_params.d_ab       # estimate of accelerometer bias drift (= uncertainty of pseudo-measurement) in each axis (std.dev. in m/s^2)
s_wb = quad_1_params.d_wb       # estimate of gyro bias drift (= uncertainty of pseudo-measurement) in each axis (std.dev. in rad/s)

s_gps = GPS_params.s_gps        # estimate of uncertainty of GPS measurement y_gps in each component (std.dev. in m)

In [ ]:
# EKF configuration

# filter_parameters (use perfect tuning)
EKF_params = GPS_only_INS_EKF.parameters()

EKF_params.s_a = s_a       # estimate of uncertainty of accelerometer measurement
EKF_params.s_w = s_w       # estimate of uncertainty of gyro measurement

EKF_params.s_ab = s_ab     # estimate of accelerometer bias drift = uncertainty of pseudo-measurement
EKF_params.s_wb = s_wb     # estimate of gyro bias drift = uncertainty of pseudo-measurement

EKF_params.s_gps = s_gps   # estimate of uncertainty of GPS measurement
        
EKF_params.g = g           # estimate of gravity vector

# filter type
EKF_config = GPS_only_INS_EKF.configuration()

In [ ]:
# EqF configuration

# filter_parameters (use perfect tuning)
EqF_params = GPS_only_INS_EqF.parameters()

EqF_params.s_a = s_a       # estimate of uncertainty of accelerometer measurement
EqF_params.s_w = s_w       # estimate of uncertainty of gyro measurement

EqF_params.s_ab = s_ab     # estimate of accelerometer bias drift = uncertainty of pseudo-measurement
EqF_params.s_wb = s_wb     # estimate of gyro bias drift = uncertainty of pseudo-measurement
        
EqF_params.s_gps = s_gps   # estimate of uncertainty of GPS measurement
        
EqF_params.g = g           # estimate of gravity vector

# filter type
EqF_config = GPS_only_INS_EqF.configuration()

## Robot definition

In [ ]:
# simple_quad robot quad_1
quad_1_name = 'quad_1'
quad_1_color = 'rgb(0,0,0)'

# sensors

IMU_runtime_attributes = {
    'g': g,
}
quad_1_IMU_sensor = base_IMU(IMU_params)
quad_1_IMU_sensor.set_runtime_attributes('IMU', quad_1_color, IMU_runtime_attributes)

quad_1_GPS_sensor = simple_GPS(GPS_params)
quad_1_GPS_sensor.set_runtime_attributes('GPS', quad_1_color, {})

quad_1_sensors = (quad_1_IMU_sensor, quad_1_GPS_sensor)

# filters
quad_1_EKF = GPS_only_INS_EKF(EKF_params, EKF_config)
name = 'EKF+bias'
color = 'rgb(0,0,255)'
quad_1_EKF.set_runtime_attributes(name, color, {})

quad_1_EqF = GPS_only_INS_EqF(EqF_params, EqF_config)
name = 'EqF'
color = 'rgb(255,0,0)'
quad_1_EqF.set_runtime_attributes(name, color, {})

quad_1_filters = (quad_1_EKF, quad_1_EqF)

# robot
quad_1 = simple_quad(quad_1_params, quad_1_config, quad_1_sensors, quad_1_filters)
quad_1.set_runtime_attributes(quad_1_name, quad_1_color, {})

## Robot and sensor testing

In [ ]:
# # initial state for simple_quad robot quad_1, time unit (seconds), spatial unit (metres)
# quad_1_p0 = quad_1.params.wayposes[0].pos
# quad_1_v0 = np.array([[0.],[0.],[0.]])
# quad_1_q0 = quad_1.params.wayposes[0].quat
# quad_1_ab0 = .05 * np.array([[1.],[1.],[1.]])
# quad_1_wb0 = np.deg2rad(.5) * np.array([[1.],[1.],[1.]])

# # acceleration and angular velocity are given in the body-fixed frame
# R = quat_utils.quat2Rot(quad_1_q0)
# quad_1_a0 = -R.T @ g
# quad_1_w0= np.array([[0.],[0.],[0.]])

# quad_1_s0 = INS_robot_state(quad_1_p0, quad_1_v0, quad_1_q0, quad_1_ab0, quad_1_wb0, quad_1_a0, quad_1_w0)

# # precompute robot trajectory and all measurements
# dt = 0.01
# quad_1.set_dt(dt)
# quad_1.precompute_trajectory(quad_1_s0)
# m_traj, m_true_traj, m_e_traj = quad_1.measure_all()

In [ ]:
# from plots.plotly_plots import plotter

# plt = plotter()
# plt.plot_robot_state(quad_1, 'p', dt)
# #plt.plot_measurement(quad_1, 'a', dt, m_traj, m_true_traj, m_e_traj)
# #plt.plot_measurement(quad_1, 'w', dt, m_traj, m_true_traj, m_e_traj)
# #plt.plot_measurement(quad_1, 'p', dt, m_traj, m_true_traj, m_e_traj)

## Simulation configuration

In [ ]:
# initial state for simple_quad robot quad_1, time unit (seconds), spatial unit (metres)
quad_1_p0 = quad_1.params.wayposes[0].pos
quad_1_v0 = np.array([[0.],[0.],[0.]])
quad_1_q0 = quad_1.params.wayposes[0].quat
quad_1_ab0 = .05 * np.array([[1.],[1.],[1.]])
quad_1_wb0 = np.deg2rad(.5) * np.array([[1.],[1.],[1.]])

# acceleration and angular velocity are given in the body-fixed frame
R = quat_utils.quat2Rot(quad_1_q0)
quad_1_a0 = -R.T @ g
quad_1_w0= np.array([[0.],[0.],[0.]])

quad_1_s0 = INS_robot_state(quad_1_p0, quad_1_v0, quad_1_q0, quad_1_ab0, quad_1_wb0, quad_1_a0, quad_1_w0)

In [ ]:
# robot data for simulation: list of pairs of the form (robot, initial state)
robot_data = [
    (quad_1, quad_1_s0)
]

In [ ]:
# configure simulation parameters 
sim_params = GPS_only_INS_sim.parameters()

sim_params.freq = 100.0                 # fixed simulation frequency (in Hz)

sim_params.s_p = [1.0 for _ in robot_data] # uncertainty of initial robot position estimate
sim_params.s_v = [0.1 for _ in robot_data] # uncertainty of initial robot velocity estimate in each compnent (std.dev. in m/s)
sim_params.s_th = [np.deg2rad(5.0) for _ in robot_data] # uncertainty of initial robot heading estimate# uncertainty of initial robot heading estimate
sim_params.s_ab = [0.01 for _ in robot_data] # uncertainty of initial accelerometer bias estimate in each axis (std.dev. in m/s^2)
sim_params.s_wb = [np.deg2rad(0.1) for _ in robot_data] # uncertainty of initial gyro bias estimate in each axis (std.dev. in rad/s)

sim_params.Pr0 = [
    np.diag(np.array([sim_params.s_p[0]**2, sim_params.s_p[0]**2, sim_params.s_p[0]**2,     # initial covariance of robot state estimate
                      sim_params.s_v[0]**2, sim_params.s_v[0]**2, sim_params.s_v[0]**2,
                      sim_params.s_th[0]**2, sim_params.s_th[0]**2, sim_params.s_th[0]**2,
                      sim_params.s_ab[0]**2, sim_params.s_ab[0]**2, sim_params.s_ab[0]**2,
                      sim_params.s_wb[0]**2, sim_params.s_wb[0]**2, sim_params.s_wb[0]**2]))
    for _ in robot_data]

# configuration flags for simulation
sim_config = GPS_only_INS_sim.configuration()

sim_config.precompute_robot_trajectories = True

sim_config.record_inival = False        # this can be overridden by argument enable_replay to sim_run()
sim_config.record_measurement = False   # this can be overridden by argument enable_replay to sim_run()

sim_config.record_final_error = True
sim_config.record_time_averaged_error = True

## Simulation and filter testing

In [ ]:
# # initialize simulation
# sim = GPS_only_INS_sim(sim_params, sim_config, robot_data)

# # run filter simulation (single run)
# result_data = sim.sim_run(0, enable_replay = True)
# replay_data = sim.replay(result_data, record_pos_error_ellipsoid = False)

In [ ]:
# from plots.plotly_plots import plotter

# plt = plotter()
# #plt.plot_filter_state(quad_1, 'p', sim.dt, replay_data, 0)
# plt.plot_filter_state(quad_1, 'q', sim.dt, replay_data, 0)

## Monte Carlo filter simulation 

In [ ]:
# more package imports
import multiprocessing as mp
from functools import partial
import tqdm as tqdm
import shelve

In [ ]:
# Monte Carlo simulation

# initialize simulation
sim = GPS_only_INS_sim(sim_params, sim_config, robot_data)

def run(sim: GPS_only_INS_sim, result_data_file: str, enable_replay: bool, 
        n_MC: int, n_proc: int, n_reuse: int):

    # open data shelf, OVERWRITES PREVIOUS FILE!!!
    data = shelve.open(result_data_file, flag = 'n')

    # shelve any global data needed for replaying/debugging
    data['sim'] = sim
    data['n_MC'] = n_MC
    
    # initialize result lists, one sublist per filter
    if sim.config.record_final_error:
        e_N = [[[None] * n_MC for _ in robot.filters] for robot in sim.robots]
    if sim.config.record_time_averaged_error:
        e_avg = [[[None] * n_MC for _ in robot.filters] for robot in sim.robots]

    # parallelized simulation loop
    with mp.Pool(n_proc, maxtasksperchild = n_reuse) as p, tqdm.tqdm(total=n_MC) as pbar:
        for result_data in p.imap_unordered(partial(sim.sim_run, enable_replay = enable_replay), range(n_MC)):
            # update the progress bar
            pbar.update()
            pbar.refresh()
                    
            # shelve the result data
            run_id = result_data['run_id']
            data['result_data[{}]'.format(run_id)] = result_data

            # repack all data where we want the top slice to be the robot+filter index
            if sim.config.record_final_error:
                for r_idx, robot in enumerate(sim.robots):
                    for f_idx in range(len(robot.filters)):
                        e_N[r_idx][f_idx][run_id] = result_data['e_N'][r_idx][f_idx]
            
            if sim.config.record_time_averaged_error:
                for r_idx, robot in enumerate(sim.robots):
                    for f_idx in range(len(robot.filters)):
                        e_avg[r_idx][f_idx][run_id] = result_data['e_avg'][r_idx][f_idx]

    # shelve the repacked data
    if sim.config.record_final_error:
        data['e_N'] = tuple(e_N)

    if sim.config.record_time_averaged_error:
        data['e_avg'] = tuple(e_avg)

    # close the data shelf
    data.close()

# CONFIGURE HERE

result_data_file = 'data/result_data'

# set enable_replay to true to record filter trajectories
enable_replay = True

# number of Monte Carlo runs
n_MC = 10

# number of parallel processes
n_proc = 16

# max times a child process can be reused before resetting (stops ballooning memory consumption)
n_reuse = 5

run(sim, result_data_file, enable_replay, n_MC, n_proc, n_reuse)

## Test plot Monte Carlo simulation results

In [ ]:
# import shelve

# from plots.plotly_plots import plotter

# # result data file, change this to read different saved/renamed result data files
# data_file = 'data/result_data'

# # read first robot and its error_data from data shelve
# data = shelve.open(data_file, flag = 'r')
# sim = data['sim']
# robot = sim.robots[0]
# error_type = 'e_N'
# error_data = data[error_type][0]
# data.close()

# plt = plotter()
# plt.plot_error_histogram(robot, sim, error_type, 'p', error_data)
# #plt.plot_error_histogram(robot, sim, error_type, 'q', error_data)

## Interactive app

In [ ]:
from UI.simple_app import simple_app

# result data file, change this to read different saved/renamed result data files
data_file = 'data/result_data'

# where should the app buffer the downloadable file
downloadable_file_buffer = 'data/single_run'

# create the app
dash_app = simple_app(data_file, downloadable_file_buffer)

# start app and display in external browser tab
dash_app.app.run(jupyter_mode = 'external')